# P3 — Gradient Descent (GD)

Placement Prediction System — Gradient Descent implementation.

This notebook uses the CSV in the project's `data` folder, automatically cleans column names, detects a suitable numeric target, scales features, and implements Gradient Descent from scratch.

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. Load the placement dataset

In [ ]:
DATA_DIR = r"C:\Users\tallu\Downloads\Placement-Prediction System\data"

csv_files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
if not csv_files:
    raise FileNotFoundError(f'No CSV files found in {DATA_DIR}')

preferred = [f for f in csv_files if 'placement' in os.path.basename(f).lower()]
CSV_PATH = preferred[0] if preferred else csv_files[0]

print("Using:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
df.columns = [str(c).strip() for c in df.columns]
print('Shape:', df.shape)
display(df.head())
print('\nColumns:')
print(df.columns.tolist())


## 2. Select the target column

GD requires a numeric target for this regression example. The notebook first looks for common salary/package names. If none is found, it selects the last numeric column.

In [ ]:
def norm(s):
    return ''.join(ch.lower() for ch in str(s) if ch.isalnum())

target_names = [
    'Salary Package', 'Salary_Package', 'SalaryPackage',
    'Package', 'Salary', 'Salary_LPA', 'Package_LPA', 'CTC', 'LPA'
]

TARGET = None
lookup = {norm(c): c for c in df.columns}
for name in target_names:
    if norm(name) in lookup:
        TARGET = lookup[norm(name)]
        break

if TARGET is None:
    for c in df.columns:
        if any(x in norm(c) for x in ['salary', 'package', 'ctc', 'lpa']):
            test = pd.to_numeric(df[c], errors='coerce')
            if test.notna().sum() > 0:
                TARGET = c
                break

if TARGET is None:
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        raise ValueError('No numeric target column could be detected. Set TARGET manually.')
    TARGET = numeric_cols[-1]

print('Target column:', TARGET)


## 3. Prepare numerical features

In [ ]:
# Convert target to numeric
df[TARGET] = pd.to_numeric(
    df[TARGET].astype(str).str.replace(',', '', regex=False)
      .str.extract(r'([-+]?\d*\.?\d+)')[0],
    errors='coerce'
)
df = df.dropna(subset=[TARGET]).copy()

# Keep numeric predictors only for the from-scratch GD implementation.
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
feature_cols = [c for c in numeric_cols if c != TARGET]

# Remove obvious ID columns
feature_cols = [c for c in feature_cols if norm(c) not in {
    'id', 'studentid', 'studentno', 'studentnumber', 'rollno', 'rollnumber'
} and not norm(c).endswith('id')]

if not feature_cols:
    raise ValueError('No numeric predictor columns were found for GD.')

X = df[feature_cols].copy()
y = df[TARGET].to_numpy(dtype=float).reshape(-1, 1)

# Fill missing feature values with column medians
X = X.fillna(X.median(numeric_only=True))

print('Features:', feature_cols)
print('Target:', TARGET)
display(X.head())


## 4. Standardize features

In [ ]:
X_values = X.to_numpy(dtype=float)
X_mean = X_values.mean(axis=0)
X_std = X_values.std(axis=0)
X_std[X_std == 0] = 1

X_scaled = (X_values - X_mean) / X_std

# Add intercept column
X_gd = np.c_[np.ones((X_scaled.shape[0], 1)), X_scaled]

print('X shape after adding intercept:', X_gd.shape)


## 5. Gradient Descent from scratch

Model: `y_hat = Xθ`

Cost: `J(θ) = (1/(2m)) Σ(y_hat - y)²`

Update: `θ = θ - α (1/m) Xᵀ(Xθ - y)`

In [ ]:
def gradient_descent(X, y, learning_rate=0.01, epochs=1000):
    m, n = X.shape
    theta = np.zeros((n, 1))
    cost_history = []

    for epoch in range(epochs):
        y_pred = X @ theta
        error = y_pred - y
        cost = (1 / (2 * m)) * np.sum(error ** 2)
        cost_history.append(cost)

        gradient = (1 / m) * (X.T @ error)
        theta = theta - learning_rate * gradient

    return theta, cost_history

learning_rate = 0.01
epochs = 2000

theta, cost_history = gradient_descent(
    X_gd, y,
    learning_rate=learning_rate,
    epochs=epochs
)

print('Gradient Descent completed.')
print('Initial cost:', cost_history[0])
print('Final cost:', cost_history[-1])


## 6. Display learned parameters

In [ ]:
parameter_names = ['Intercept'] + feature_cols
parameters = pd.DataFrame({'Parameter': parameter_names, 'Theta': theta.ravel()})
display(parameters)


## 7. Cost function convergence

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, epochs + 1), cost_history)
plt.xlabel('Epoch')
plt.ylabel('Cost (MSE-based)')
plt.title('Gradient Descent Convergence')
plt.grid(True)
plt.show()


## 8. Predictions and evaluation

In [ ]:
y_pred = X_gd @ theta
error = y_pred - y

mae = np.mean(np.abs(error))
mse = np.mean(error ** 2)
rmse = np.sqrt(mse)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r2 = 1 - (ss_res / ss_tot) if ss_tot != 0 else np.nan

print('Gradient Descent Results')
print('------------------------')
print(f'MAE  : {mae:.4f}')
print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'R²   : {r2:.4f}')

results = pd.DataFrame({
    'Actual': y.ravel(),
    'Predicted': y_pred.ravel()
})
display(results.head(10))


## 9. Actual vs Predicted

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(y, y_pred)
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title('Actual vs Predicted — Gradient Descent')
plt.grid(True)
plt.show()


## 10. Try different learning rates

In [ ]:
learning_rates = [0.001, 0.01, 0.05]

for lr in learning_rates:
    _, history = gradient_descent(X_gd, y, learning_rate=lr, epochs=1000)
    print(f'Learning rate {lr}: final cost = {history[-1]:.4f}')
